In [ ]:

%run input/Format.ipynb
import ROOT as root
root.gErrorIgnoreLevel = root.kFatal
%jsroot on
root.gStyle.SetOptStat(0)
root.gStyle.SetNumberContours(50)


In [ ]:

safe_to_pdf = True
outdir = "output/duplicate"
root.gSystem.mkdir(outdir, True)

file_path = "input/"
file_name = "doublecount.root"

data_set = "p+p 2025"
run_text = "Run 79513"

cuts_text = "n_{TPC clusters} #geq 30, |z_{PCA}| < 10 cm, p_{T} > 0.2 GeV/c"


In [ ]:

hist_names = [
    "h_dpx_dpy_dpz",
    "h_dphi_dpt_dtheta",
    "h_dphi_dpt",
    "h_dphi_relDpt",
    "h_deltaP_relDpt",
    "h_deltaP_dtheta",
    "h_deltaP",
    "h_deltaPhi",
    "h_deltaPt",
    "h_relDeltaPt",
    "h_deltaTheta",
    "h_duplicateScore",
    "h_trackMultiplicity",
    "h_trackPairsPerEvent",
    "h_closePairsPerEvent"
]

input_file = root.TFile.Open(file_path+file_name)

if not input_file or input_file.IsZombie():
    raise RuntimeError("Cannot open "+file_path+file_name)

hists = {}

for hist_name in hist_names:
    h = input_file.Get(hist_name)

    if not h:
        print("Missing histogram:", hist_name)
        continue

    h.SetDirectory(0)
    h.SetName(hist_name+"_plot")
    hists[hist_name] = h

input_file.Close()

print("Loaded", len(hists), "histograms")
print(list(hists.keys()))


In [ ]:

def Format_hist_loc(h, title, x_title, y_title, x_title_size=0.06, y_title_size=0.06,
                    x_title_offset=1.0, y_title_offset=1.0,
                    label_size_x=0.06, label_size_y=0.06, label_size_z=0.05,
                    z_axis_offset=1.0, z_range=None, center_title=False):

    if h is None:
        print("Error: Histogram is None. Cannot format.")
        return

    if title is not None:
        h.SetTitle(title)

    if x_title is not None:
        h.GetXaxis().SetTitle(x_title)

    if y_title is not None:
        h.GetYaxis().SetTitle(y_title)

    h.GetXaxis().SetTitleSize(x_title_size)
    h.GetYaxis().SetTitleSize(y_title_size)

    h.GetXaxis().SetTitleOffset(x_title_offset)
    h.GetYaxis().SetTitleOffset(y_title_offset)

    h.GetXaxis().SetLabelSize(label_size_x)
    h.GetYaxis().SetLabelSize(label_size_y)
    h.GetZaxis().SetLabelSize(label_size_z)
    h.GetZaxis().SetTitleOffset(z_axis_offset)

    if z_range is not None:
        h.GetZaxis().SetRangeUser(z_range[0], z_range[1])

    if center_title:
        h.GetXaxis().CenterTitle()
        h.GetYaxis().CenterTitle()


def Draw_header(extra_text=None, x=0.15):

    latex = root.TLatex()
    latex.SetNDC()
    latex.SetTextFont(42)

    latex.SetTextSize(0.052)
    latex.DrawLatex(x, 0.93, data_set+", "+run_text)

    latex.SetTextSize(0.041)
    latex.DrawLatex(x, 0.875, cuts_text)

    if extra_text is not None:
        latex.DrawLatex(x, 0.825, extra_text)

    return latex


def Set_pad_2d():
    root.gPad.SetLeftMargin(0.12)
    root.gPad.SetBottomMargin(0.12)
    root.gPad.SetRightMargin(0.15)
    root.gPad.SetTopMargin(0.10)
    root.gPad.SetLogz()


def Set_pad_1d():
    root.gPad.SetLeftMargin(0.12)
    root.gPad.SetBottomMargin(0.12)
    root.gPad.SetRightMargin(0.04)
    root.gPad.SetTopMargin(0.10)
    root.gPad.SetLogy()


In [ ]:

#main duplicate-sensitive 2D plots

plots_2d = [
    ("h_dphi_relDpt",
     "#Delta#phi [rad]",
     "|#Deltap_{T}|/#LTp_{T}#GT",
     "Same-event track pairs",
     (-0.05, 0.05),
     (0.0, 0.15)),

    ("h_dphi_dpt",
     "#Delta#phi [rad]",
     "#Deltap_{T} [GeV/c]",
     "Same-event track pairs",
     (-0.05, 0.05),
     (-0.20, 0.20)),

    ("h_deltaP_relDpt",
     "|#Delta#vec{p}| [GeV/c]",
     "|#Deltap_{T}|/#LTp_{T}#GT",
     "Same-event track pairs",
     (0.0, 0.30),
     (0.0, 0.15)),

    ("h_deltaP_dtheta",
     "|#Delta#vec{p}| [GeV/c]",
     "#Delta#theta [rad]",
     "Same-event track pairs",
     (0.0, 0.30),
     (-0.05, 0.05))
]

canvases_2d = []

for hist_name, xtitle, ytitle, plot_text, xrange, yrange in plots_2d:

    if hist_name not in hists:
        continue

    c = root.TCanvas("c_"+hist_name, "c_"+hist_name, 1100, 900)
    Set_pad_2d()

    h = hists[hist_name]

    Format_hist_loc(
        h,
        title="",
        x_title=xtitle,
        y_title=ytitle,
        x_title_offset=.9,
        y_title_offset=.9,
        center_title=True,
        x_title_size=0.06,
        y_title_size=0.06,
        label_size_x=0.05,
        label_size_y=0.05,
        label_size_z=0.045,
        z_axis_offset=0.85
    )

    h.GetXaxis().SetRangeUser(xrange[0], xrange[1])
    h.GetYaxis().SetRangeUser(yrange[0], yrange[1])
    h.Draw("COLZ")

    latex = Draw_header(plot_text)

    c.Draw()
    canvases_2d.append(c)

    if safe_to_pdf:
        c.SaveAs(outdir+"/"+hist_name+".pdf")


In [ ]:

#close-up views around the duplicate-track region

close_plots = [
    ("h_dphi_relDpt",
     "#Delta#phi [rad]",
     "|#Deltap_{T}|/#LTp_{T}#GT",
     (-0.01, 0.01),
     (0.0, 0.03)),

    ("h_dphi_dpt",
     "#Delta#phi [rad]",
     "#Deltap_{T} [GeV/c]",
     (-0.01, 0.01),
     (-0.05, 0.05)),

    ("h_deltaP_relDpt",
     "|#Delta#vec{p}| [GeV/c]",
     "|#Deltap_{T}|/#LTp_{T}#GT",
     (0.0, 0.05),
     (0.0, 0.03)),

    ("h_deltaP_dtheta",
     "|#Delta#vec{p}| [GeV/c]",
     "#Delta#theta [rad]",
     (0.0, 0.05),
     (-0.01, 0.01))
]

canvases_close = []

for hist_name, xtitle, ytitle, xrange, yrange in close_plots:

    if hist_name not in hists:
        continue

    c = root.TCanvas("c_"+hist_name+"_close", "c_"+hist_name+"_close", 1100, 900)
    Set_pad_2d()

    h = hists[hist_name].Clone(hist_name+"_close")
    h.SetDirectory(0)

    Format_hist_loc(
        h,
        title="",
        x_title=xtitle,
        y_title=ytitle,
        x_title_offset=.9,
        y_title_offset=.9,
        center_title=True,
        x_title_size=0.06,
        y_title_size=0.06,
        label_size_x=0.05,
        label_size_y=0.05,
        label_size_z=0.045,
        z_axis_offset=0.85
    )

    h.GetXaxis().SetRangeUser(xrange[0], xrange[1])
    h.GetYaxis().SetRangeUser(yrange[0], yrange[1])
    h.Draw("COLZ")

    latex = Draw_header("Zoom near duplicate-track region")

    c.Draw()
    canvases_close.append(c)

    if safe_to_pdf:
        c.SaveAs(outdir+"/"+hist_name+"_close.pdf")


In [ ]:

#2D projections of h_dpx_dpy_dpz

if "h_dpx_dpy_dpz" in hists:

    h3_p = hists["h_dpx_dpy_dpz"]

    projections_p = [
        ("yx", "#Deltap_{x} [GeV/c]", "#Deltap_{y} [GeV/c]", "h_dpx_dpy"),
        ("zx", "#Deltap_{x} [GeV/c]", "#Deltap_{z} [GeV/c]", "h_dpx_dpz"),
        ("zy", "#Deltap_{y} [GeV/c]", "#Deltap_{z} [GeV/c]", "h_dpy_dpz")
    ]

    canvases_p = []
    projected_p = []

    for option, xtitle, ytitle, output_name in projections_p:

        h2 = h3_p.Project3D(option)
        h2.SetName(output_name)
        h2.SetDirectory(0)
        projected_p.append(h2)

        c = root.TCanvas("c_"+output_name, "c_"+output_name, 1100, 900)
        Set_pad_2d()

        Format_hist_loc(
            h2,
            title="",
            x_title=xtitle,
            y_title=ytitle,
            x_title_offset=.9,
            y_title_offset=.9,
            center_title=True,
            x_title_size=0.06,
            y_title_size=0.06,
            label_size_x=0.05,
            label_size_y=0.05,
            label_size_z=0.045,
            z_axis_offset=0.85
        )

        h2.GetXaxis().SetRangeUser(-0.10, 0.10)
        h2.GetYaxis().SetRangeUser(-0.10, 0.10)
        h2.Draw("COLZ")

        latex = Draw_header("Same-event momentum-difference pairs")

        c.Draw()
        canvases_p.append(c)

        if safe_to_pdf:
            c.SaveAs(outdir+"/"+output_name+".pdf")


In [ ]:

#close-up momentum-difference projections

if "h_dpx_dpy_dpz" in hists:

    close_projected_p = []
    close_canvases_p = []

    for option, xtitle, ytitle, output_name in projections_p:

        h2 = h3_p.Project3D(option)
        h2.SetName(output_name+"_close")
        h2.SetDirectory(0)
        close_projected_p.append(h2)

        c = root.TCanvas("c_"+output_name+"_close", "c_"+output_name+"_close", 1100, 900)
        Set_pad_2d()

        Format_hist_loc(
            h2,
            title="",
            x_title=xtitle,
            y_title=ytitle,
            x_title_offset=.9,
            y_title_offset=.9,
            center_title=True,
            x_title_size=0.06,
            y_title_size=0.06,
            label_size_x=0.05,
            label_size_y=0.05,
            label_size_z=0.045,
            z_axis_offset=0.85
        )

        h2.GetXaxis().SetRangeUser(-0.02, 0.02)
        h2.GetYaxis().SetRangeUser(-0.02, 0.02)
        h2.Draw("COLZ")

        latex = Draw_header("Zoom near #Delta#vec{p}=0")

        c.Draw()
        close_canvases_p.append(c)

        if safe_to_pdf:
            c.SaveAs(outdir+"/"+output_name+"_close.pdf")


In [ ]:

#2D projections of h_dphi_dpt_dtheta

if "h_dphi_dpt_dtheta" in hists:

    h3_ang = hists["h_dphi_dpt_dtheta"]

    projections_ang = [
        ("yx", "#Delta#phi [rad]", "#Deltap_{T} [GeV/c]", "h_dphi_dpt_from3d"),
        ("zx", "#Delta#phi [rad]", "#Delta#theta [rad]", "h_dphi_dtheta"),
        ("zy", "#Deltap_{T} [GeV/c]", "#Delta#theta [rad]", "h_dpt_dtheta")
    ]

    canvases_ang = []
    projected_ang = []

    for option, xtitle, ytitle, output_name in projections_ang:

        h2 = h3_ang.Project3D(option)
        h2.SetName(output_name)
        h2.SetDirectory(0)
        projected_ang.append(h2)

        c = root.TCanvas("c_"+output_name, "c_"+output_name, 1100, 900)
        Set_pad_2d()

        Format_hist_loc(
            h2,
            title="",
            x_title=xtitle,
            y_title=ytitle,
            x_title_offset=.9,
            y_title_offset=.9,
            center_title=True,
            x_title_size=0.06,
            y_title_size=0.06,
            label_size_x=0.05,
            label_size_y=0.05,
            label_size_z=0.045,
            z_axis_offset=0.85
        )

        h2.Draw("COLZ")
        latex = Draw_header("Same-event angular and momentum-difference pairs")

        c.Draw()
        canvases_ang.append(c)

        if safe_to_pdf:
            c.SaveAs(outdir+"/"+output_name+".pdf")


In [ ]:

#one-dimensional duplicate-sensitive distributions in a 2x3 canvas

one_d_names = [
    ("h_deltaP", "|#Delta#vec{p}| [GeV/c]"),
    ("h_deltaPhi", "#Delta#phi [rad]"),
    ("h_deltaPt", "#Deltap_{T} [GeV/c]"),
    ("h_relDeltaPt", "|#Deltap_{T}|/#LTp_{T}#GT"),
    ("h_deltaTheta", "#Delta#theta [rad]"),
    ("h_duplicateScore", "duplicate-track score")
]

c_1d = root.TCanvas("c_duplicate_1d", "c_duplicate_1d", 1500, 1000)
c_1d.Divide(3, 2)

one_d_drawn = []

for i, (hist_name, xtitle) in enumerate(one_d_names):

    if hist_name not in hists:
        continue

    c_1d.cd(i+1)

    root.gPad.SetLeftMargin(0.13)
    root.gPad.SetBottomMargin(0.13)
    root.gPad.SetRightMargin(0.04)
    root.gPad.SetTopMargin(0.10)
    root.gPad.SetLogy()

    h = hists[hist_name]

    Format_hist_loc(
        h,
        title="",
        x_title=xtitle,
        y_title="pairs",
        x_title_offset=.9,
        y_title_offset=.9,
        center_title=True,
        x_title_size=0.055,
        y_title_size=0.055,
        label_size_x=0.045,
        label_size_y=0.045
    )

    h.SetLineColor(root.kBlack)
    h.SetLineWidth(2)
    h.SetMarkerColor(root.kBlack)
    h.SetMarkerStyle(20)
    h.SetMarkerSize(0.7)
    h.SetMinimum(0.5)
    h.Draw("E")

    latex = root.TLatex()
    latex.SetNDC()
    latex.SetTextFont(42)
    latex.SetTextSize(0.044)
    latex.DrawLatex(0.16, 0.92, run_text)

    one_d_drawn.append(h)

c_1d.Draw()

if safe_to_pdf:
    c_1d.SaveAs(outdir+"/duplicate_sensitive_1d.pdf")


In [ ]:

#event-level QA plots

event_hists = [
    ("h_trackMultiplicity", "selected tracks per event"),
    ("h_trackPairsPerEvent", "track pairs per event"),
    ("h_closePairsPerEvent", "duplicate-like pairs per event")
]

c_event = root.TCanvas("c_duplicate_event_qa", "c_duplicate_event_qa", 1500, 500)
c_event.Divide(3, 1)

event_drawn = []

for i, (hist_name, xtitle) in enumerate(event_hists):

    if hist_name not in hists:
        continue

    c_event.cd(i+1)

    root.gPad.SetLeftMargin(0.14)
    root.gPad.SetBottomMargin(0.15)
    root.gPad.SetRightMargin(0.04)
    root.gPad.SetTopMargin(0.11)
    root.gPad.SetLogy()

    h = hists[hist_name]

    Format_hist_loc(
        h,
        title="",
        x_title=xtitle,
        y_title="events",
        x_title_offset=1.0,
        y_title_offset=1.0,
        center_title=True,
        x_title_size=0.055,
        y_title_size=0.055,
        label_size_x=0.045,
        label_size_y=0.045
    )

    h.SetLineColor(root.kBlack)
    h.SetLineWidth(2)
    h.SetFillStyle(0)
    h.SetMinimum(0.5)
    h.Draw("HIST")

    latex = root.TLatex()
    latex.SetNDC()
    latex.SetTextFont(42)
    latex.SetTextSize(0.044)
    latex.DrawLatex(0.17, 0.92, run_text)

    event_drawn.append(h)

c_event.Draw()

if safe_to_pdf:
    c_event.SaveAs(outdir+"/duplicate_event_qa.pdf")


In [ ]:

print("All plots were written to:", outdir)
